In [7]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.utils import *

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import lightgbm as lgbm

import optuna as op

In [11]:
# Read data files
x, y, x_test = load_processed_data()

# Testing new feature combinations based on correlation

x["RelativeTyreLife"] = (x["TyreLife"] / (x["LapNumber"] + 1))

x_test["RelativeTyreLife"] = (x_test["TyreLife"] / (x_test["LapNumber"] + 1))

x["TyreLife_Per_Stint"] = (x["TyreLife"] / (x["Stint"] + 1))

x_test["TyreLife_Per_Stint"] = (x_test["TyreLife"] / (x_test["Stint"] + 1))

x["Stint_Per_RaceProgress"] = (x["Stint"] / (x["RaceProgress"] + 1))

x_test["Stint_Per_RaceProgress"] = (x_test["Stint"] / (x_test["RaceProgress"] + 1))

x["Degredation_Per_TyreLife"] = (x["Cumulative_Degradation"] / x["TyreLife"] + 1) 

x_test["Degredation_Per_TyreLife"] = (x_test["Cumulative_Degradation"] / x_test["TyreLife"] + 1) 


# Test train split from sklearn model selection
x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Parameters generated from previous optuna model
best_params = {
    "learning_rate": 0.020541079651710786,
    "num_leaves": 83,
    "max_depth": 8,
    "min_child_samples": 89,
    "subsample": 0.9997820637792558,
    "colsample_bytree": 0.6348560759485127,
    "reg_alpha": 2.42890524945383,
    "reg_lambda": 3.852578357505773
}

model = lgbm.LGBMClassifier(
    **best_params,
    n_estimators=3000,
    objective="binary",
    metric="auc",
    random_state=42,
    verbosity=-1
)

model.fit(
    x_train,
    y_train,
    eval_set=[(x_val,y_val)],
    eval_metric="AUC",
    callbacks=[
        lgbm.early_stopping(200),
        lgbm.log_evaluation(200)
    ]
)

validation_prediction = generate_probability_predict(model, x_val)

generate_auc(y_val, validation_prediction)

predictions = generate_probability_predict(model, x_test)

generate_submission(predictions, "../submissions/lgbm_FE2.csv")


Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.94369
[400]	valid_0's auc: 0.947004
[600]	valid_0's auc: 0.948345
[800]	valid_0's auc: 0.949047
[1000]	valid_0's auc: 0.949569
[1200]	valid_0's auc: 0.949911
[1400]	valid_0's auc: 0.950128
[1600]	valid_0's auc: 0.950334
[1800]	valid_0's auc: 0.950458
[2000]	valid_0's auc: 0.950552
[2200]	valid_0's auc: 0.950629
[2400]	valid_0's auc: 0.950684
[2600]	valid_0's auc: 0.950697
[2800]	valid_0's auc: 0.950699
Early stopping, best iteration is:
[2668]	valid_0's auc: 0.950711
AUC: 0.9507


,id,PitNextLap
0,439140,0.005808
1,439141,0.006606
2,439142,0.004916
3,439143,0.239760
4,439144,0.847926
...,...,...
188160,627300,0.010576
188161,627301,0.590896
188162,627302,0.743975
188163,627303,0.869548
